In [1]:
!uname -a && lscpu | grep "Model name\|CPU(s):" && free -h
!git clone https://github.com/hkust-adsl/kubernetes-scheduler-simulator.git
%cd kubernetes-scheduler-simulator
!ls -la

Linux 94777670828c 6.6.159+ #1 SMP Thu Oct 22 03:07:54 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
NUMA node0 CPU(s):                       0,1
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.2Gi       9.1Gi       2.4Mi       2.7Gi        11Gi
Swap:             0B          0B          0B
Cloning into 'kubernetes-scheduler-simulator'...
remote: Enumerating objects: 6413, done.
remote: Total 6413 (delta 0), reused 0 (delta 0), pack-reused 6413 (from 1)
Receiving objects: 100% (6413/6413), 114.97 MiB | 19.03 MiB/s, done.
Resolving deltas: 100% (1770/1770), done.
/content/kubernetes-scheduler-simulator
total 204
drwxr-xr-x 10 root root   4096 Nov 25 04:24 .
drwxr-xr-x  1 root root   4096 Nov 25 04:24 ..
drwxr-xr-x  7 root root   4096 Nov 25 04:24 cmd
drwxr-xr-x  4 root root   4096 Nov 25 04:24 data
drwxr-xr-x  3 root ro

In [2]:
!cat README.md
!echo "=== MAKEFILE ==="
!cat Makefile
!echo "=== EXPERIMENTS & SCRIPTS ==="
!ls -la experiments scripts

# 🚀 Kubernetes Scheduler Simulator

The simulator evaluates different scheduling policies in GPU-sharing clusters.
It includes the Fragmentation Gradient Descent (FGD) policy proposed in the [USENIX ATC 2023](https://www.usenix.org/conference/atc23) paper "[Beware of Fragmentation: Scheduling GPU-Sharing Workloads with Fragmentation Gradient Descent](https://www.usenix.org/conference/atc23/presentation/weng)", along with other baseline policies (e.g., Best-fit, Dot-product, GPU Packing, GPU Clustering, Random-fit). 

## 🎬 Demo
| Step 1: Init and Run Experiments                                          |  Step 2: Result Analysis                                                  |
| ------------------------------------------------------------------------  | ------------------------------------------------------------------------- |
| [![](https://asciinema.org/a/666336.svg)](https://asciinema.org/a/666336) | [![](https://asciinema.org/a/666339.svg)](https://asciinema.org/a/666339) |


## 

In [3]:
!go version
!make
!bin/simon apply --extended-resources "gpu" \
  -f example/test-cluster-config.yaml \
  -s example/test-scheduler-config.yaml

/bin/bash: line 1: go: command not found
make: go: No such file or directory
make: go: No such file or directory
GOARCH= GOOS= go build -v -o ./bin/simon ./cmd
/bin/sh: 1: go: not found
make: *** [Makefile:14: build] Error 127
/bin/bash: line 1: bin/simon: No such file or directory


In [ ]:
%%bash
wget -q https://go.dev/dl/go1.20.4.linux-amd64.tar.gz
rm -rf /usr/local/go && tar -C /usr/local -xzf go1.20.4.linux-amd64.tar.gz
export PATH=/usr/local/go/bin:$PATH
go version
make
bin/simon apply --extended-resources "gpu" \
  -f example/test-cluster-config.yaml \
  -s example/test-scheduler-config.yaml

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator
cat data/README.md
echo "=== EXPERIMENTS README ==="
cat experiments/README.md
echo "=== RUN SCRIPTS ==="
ls -la experiments/run_scripts

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator/data

# Pastikan binary simon sudah ter-build
if [ ! -f "../bin/simon" ]; then
  export PATH=/usr/local/go/bin:$PATH
  (cd .. && make)
fi

# Buat backup prepare_input.sh dan filter hanya memproses openb_pod_list_default
cp prepare_input.sh prepare_input.sh.bak
sed -i 's/openb_pod_list_\*.csv/openb_pod_list_default.csv/g' prepare_input.sh

# Jalankan persiapan input (estimasi < 30 detik)
bash prepare_input.sh

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator
export PATH=/usr/local/go/bin:$PATH
time (
  EXPDIR="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"
  mkdir -p ${EXPDIR}
  touch "${EXPDIR}/terminal.out"
  python3 scripts/generate_config_and_run.py -d "${EXPDIR}" -e -b -f data/openb_pod_list_default -FGD 1000 -gpusel FGD -dimext share -norm max -tune 1.3 -tuneseed 42 --shuffle-pod=true -z "${EXPDIR}/snapshot/ds01" | tee -a "${EXPDIR}/terminal.out"
  python3 scripts/analysis.py -f -g ${EXPDIR} | tee -a "${EXPDIR}/terminal.out"
)

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator
EXPDIR="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"
ls -lh "${EXPDIR}"
tail -n 35 "${EXPDIR}/terminal.out"

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator
EXPDIR="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"

cat "${EXPDIR}"/log-*.log
echo "=== RUN ANALYSIS.PY ==="
python3 scripts/analysis.py -f -g "${EXPDIR}"
ls -lh "${EXPDIR}"

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator
export PATH=/usr/local/go/bin:$PATH

EXP_FGD="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"
echo "=== RINGKASAN LOG FGD (BAGIAN BAWAH) ==="
tail -n 25 "${EXP_FGD}"/log-*.log
echo "=== ISI analysis.csv (FGD) ==="
cat "${EXP_FGD}/analysis.csv"

echo ""
echo "=== JALANKAN BASELINE 01-Random ==="
time (
  EXP_RND="experiments/2023_0511/openb_pod_list_default/01-Random/1.3/42"
  mkdir -p "${EXP_RND}"
  python3 scripts/generate_config_and_run.py -d "${EXP_RND}" -e -b -f data/openb_pod_list_default -Random 1000 -gpusel random -tune 1.3 -tuneseed 42 --shuffle-pod=true -z "${EXP_RND}/snapshot/ds01"
  python3 scripts/analysis.py -f -g "${EXP_RND}"
  echo "=== RINGKASAN LOG RANDOM (BAGIAN BAWAH) ==="
  tail -n 25 "${EXP_RND}"/log-*.log
  echo "=== ISI analysis.csv (RANDOM) ==="
  cat "${EXP_RND}/analysis.csv"
)

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator

EXP_FGD="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"
echo "=== RINGKASAN LOG FGD ==="
tail -n 25 "${EXP_FGD}"/log-*.log
echo "=== ISI analysis.csv (FGD) ==="
cat "${EXP_FGD}/analysis.csv"

EXP_RND="experiments/2023_0511/openb_pod_list_default/01-Random/1.3/42"
echo ""
echo "=== CEK FILE DI FOLDER RANDOM ==="
ls -lh "${EXP_RND}" 2>/dev/null || echo "Folder RANDOM belum ada / belum menghasilkan log"

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator

EXP_FGD="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"
EXP_RND="experiments/2023_0511/openb_pod_list_default/01-Random/1.3/42"

python3 scripts/analysis.py -f "${EXP_RND}"
echo "=== analysis.csv (FGD) ==="
cat "${EXP_FGD}/analysis.csv"
echo "=== analysis.csv (RANDOM) ==="
cat "${EXP_RND}/analysis.csv"
echo "=== AKHIR LOG FGD ==="
grep -E "Cluster Analysis Results|MilliCpu|Memory|Gpu|MilliGpu|idle_gpu_milli|frag_gpu_milli" "${EXP_FGD}"/log-*.log | tail -n 15
echo "=== AKHIR LOG RANDOM ==="
grep -E "Cluster Analysis Results|MilliCpu|Memory|Gpu|MilliGpu|idle_gpu_milli|frag_gpu_milli" "${EXP_RND}"/log-*.log | tail -n 15

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator

EXP_FGD="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"
EXP_RND="experiments/2023_0511/openb_pod_list_default/01-Random/1.3/42"

echo "=== 20 BARIS TERAKHIR LOG FGD ==="
tail -n 20 "${EXP_FGD}"/log-*.log
echo "=== analysis_fail.out (FGD) ==="
cat "${EXP_FGD}/analysis_fail.out"

echo ""
echo "=== 20 BARIS TERAKHIR LOG RANDOM ==="
tail -n 20 "${EXP_RND}"/log-*.log
echo "=== analysis_fail.out (RANDOM) ==="
cat "${EXP_RND}/analysis_fail.out"

echo ""
echo "=== EXPECTED RESULTS (DEFAULT, TUNE 1.3, SEED 42) ==="
grep -E "workload|default" experiments/analysis/expected_results/analysis_allo_discrete.csv | grep -E "workload|1\.3" | grep -E "workload|42" | head -n 10

In [ ]:
%%bash
cd /content/kubernetes-scheduler-simulator

EXP_FGD="experiments/2023_0511/openb_pod_list_default/06-FGD/1.3/42"
wc -l "${EXP_FGD}"/log-*.log
grep -E "Cluster Analysis Results|Frag ratio|Used GPU Milli" "${EXP_FGD}"/log-*.log | tail -n 10

echo "=== BANDINGKAN 06-FGD vs 01-Random (DEFAULT, TUNE 1.3, SEED 42) ==="
grep "openb_pod_list_default," experiments/analysis/expected_results/analysis_allo_discrete.csv | grep ",1\.3,42," | awk -F',' '{print $2, "-> awal:", $6, "akhir(130):", $NF}'